# 语义切分全流程调试分析

分析为什么语义切分只产生一块的问题。


In [3]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# 添加项目路径
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

# 加载环境变量
env_path = project_root / "configs" / "env"
if env_path.exists():
    load_dotenv(env_path)

from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from wxchatrag.chunking import ChunkStrategy
from wxchatrag.embeddings.bge_embeddings import BGEEmbeddings
from wxchatrag.settings import get_settings
import numpy as np
import inspect
import re


## 1. 准备测试文档


In [4]:
# 使用一个较长的中文测试文档
test_text = """
人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。

机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编程。深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作方式。

自然语言处理（NLP）是人工智能的另一个重要领域，它使计算机能够理解、解释和生成人类语言。计算机视觉是人工智能的一个分支，它使计算机能够从图像和视频中提取有意义的信息。

强化学习是一种机器学习方法，其中智能体通过与环境交互来学习最优行为策略。这些技术正在改变我们生活的方方面面，从医疗诊断到自动驾驶汽车，从智能助手到推荐系统。

在人工智能的发展历程中，有几个重要的里程碑。1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。1956年，达特茅斯会议标志着人工智能学科的正式诞生。

随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。

然而，人工智能也面临着诸多挑战。数据隐私和安全问题、算法的公平性和透明度、以及人工智能对就业市场的影响，都是需要认真考虑的问题。

未来，人工智能将继续发展，可能会在更多领域发挥重要作用。通用人工智能（AGI）的实现仍然是一个长期目标，需要更多的研究和创新。
"""

test_doc = Document(
    page_content=test_text.strip(),
    metadata={"source": "test_document", "title": "AI概述"}
)

print(f"文档长度: {len(test_doc.page_content)} 字符")
print(f"文档内容:\n{test_doc.page_content[:500]}...")


文档长度: 576 字符
文档内容:
人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。

机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编程。深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作方式。

自然语言处理（NLP）是人工智能的另一个重要领域，它使计算机能够理解、解释和生成人类语言。计算机视觉是人工智能的一个分支，它使计算机能够从图像和视频中提取有意义的信息。

强化学习是一种机器学习方法，其中智能体通过与环境交互来学习最优行为策略。这些技术正在改变我们生活的方方面面，从医疗诊断到自动驾驶汽车，从智能助手到推荐系统。

在人工智能的发展历程中，有几个重要的里程碑。1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。1956年，达特茅斯会议标志着人工智能学科的正式诞生。

随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。

然而，人工智能也面临着诸多挑战。数据隐私和安全问题、算法的公平性和透明度、以及人工智能对就业市场的影响，都...


## 2. 检查 SemanticChunker 的参数


In [5]:
# 查看 SemanticChunker 的初始化参数
print("SemanticChunker 的初始化参数:")
print(inspect.signature(SemanticChunker.__init__))

# 创建embedding实例
settings = get_settings()
embeddings = BGEEmbeddings(model_name=settings.semantic_embedding_model_name)
print(f"\n使用embedding模型: {settings.semantic_embedding_model_name}")


SemanticChunker 的初始化参数:
(self, embeddings: langchain_core.embeddings.embeddings.Embeddings, buffer_size: int = 1, add_start_index: bool = False, breakpoint_threshold_type: Literal['percentile', 'standard_deviation', 'interquartile', 'gradient'] = 'percentile', breakpoint_threshold_amount: float | None = None, number_of_chunks: int | None = None, sentence_split_regex: str = '(?<=[.?!])\\s+', min_chunk_size: int | None = None)


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 915.57it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



使用embedding模型: BAAI/bge-small-zh-v1.5


## 3. 测试不同分句模式与切分结果


In [6]:
# 定义分句模式与辅助函数
sentence_split_presets = {
    "english": r"(?<=[.?!])\s+",
    "chinese": r"(?<=[。！？；])|\n+",
    "mixed": r"(?<=[。！？；])|(?<=[.?!])\s+|\n+",
    "custom": r"(?<=[。！？；：])|(?<=[.?!:;])\s+|\n+",
}


def compact_sentences(items):
    return [item.strip() for item in items if item and item.strip()]


def run_semantic_chunker(doc, *, mode, regex, min_chunk_size=50, threshold_type="percentile"):
    splitter = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type=threshold_type,
        add_start_index=True,
        min_chunk_size=min_chunk_size,
        sentence_split_regex=regex,
    )
    chunks = splitter.split_documents([doc])
    return chunks


print("可测试的分句模式:")
for mode, regex in sentence_split_presets.items():
    print(f"- {mode}: {regex}")


可测试的分句模式:
- english: (?<=[.?!])\s+
- chinese: (?<=[。！？；])|\n+
- mixed: (?<=[。！？；])|(?<=[.?!])\s+|\n+
- custom: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+


In [7]:
# 先直接看不同 regex 的原始分句效果
print("=" * 60)
print("原始正则分句对比")
print("=" * 60)

sentence_stats = {}
for mode, regex in sentence_split_presets.items():
    raw_sentences = re.split(regex, test_doc.page_content)
    sentences = compact_sentences(raw_sentences)
    sentence_stats[mode] = sentences
    print(f"\n[{mode}]")
    print(f"regex: {regex}")
    print(f"句子数: {len(sentences)}")
    for i, sentence in enumerate(sentences[:5], 1):
        print(f"  {i}. {sentence[:60]}")


原始正则分句对比

[english]
regex: (?<=[.?!])\s+
句子数: 1
  1. 人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。

机器学习是人工智能的一个子集，

[chinese]
regex: (?<=[。！？；])|\n+
句子数: 16
  1. 人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。
  2. 机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编程。
  3. 深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作方式。
  4. 自然语言处理（NLP）是人工智能的另一个重要领域，它使计算机能够理解、解释和生成人类语言。
  5. 计算机视觉是人工智能的一个分支，它使计算机能够从图像和视频中提取有意义的信息。

[mixed]
regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+
句子数: 16
  1. 人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。
  2. 机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编程。
  3. 深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作方式。
  4. 自然语言处理（NLP）是人工智能的另一个重要领域，它使计算机能够理解、解释和生成人类语言。
  5. 计算机视觉是人工智能的一个分支，它使计算机能够从图像和视频中提取有意义的信息。

[custom]
regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+
句子数: 16
  1. 人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。
  2. 机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编程。
  3. 深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作方式。
  4. 自然语言处理（NLP）是人工智能的另一个重要领域，它使计算机能够理解、解释和生成人类语言。
  5. 计算机视觉是人工智能的一个分支，它使计算机能够从图像和视频中提取有意义的信息。


In [8]:
# 直接测试 SemanticChunker 在不同分句模式下的行为
print("=" * 60)
print("SemanticChunker 分句模式对比")
print("=" * 60)

mode_results = {}
for mode, regex in sentence_split_presets.items():
    chunks = run_semantic_chunker(
        test_doc,
        mode=mode,
        regex=regex,
        min_chunk_size=50,
        threshold_type="percentile",
    )
    mode_results[mode] = chunks
    print(f"\n[{mode}]")
    print(f"块数量: {len(chunks)}")
    print(f"块长度列表: {[len(chunk.page_content) for chunk in chunks]}")
    for i, chunk in enumerate(chunks[:3], 1):
        print(f"  块 {i}: {chunk.page_content[:80]!r}")


SemanticChunker 分句模式对比

[english]
块数量: 1
块长度列表: [576]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。\n\n机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'

[chinese]
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'
  块 2: '在人工智能的发展历程中，有几个重要的里程碑。 1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。 1956年，达特茅斯会议标志着人工智能学科'
  块 3: '随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。 深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。  然'

[mixed]
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'
  块 2: '在人工智能的发展历程中，有几个重要的里程碑。 1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。 1956年，达特茅斯会议标志着人工智能学科'
  块 3: '随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。 深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。  然'

[custom]
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'
  块 2: '在人工智能的发展历程中，有几个重要的里程碑。 1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。 1956年，达特茅斯会议标志着人工智能学科'
  块 3: '随着计算能力的提升

In [9]:
# 在 mixed 模式下继续比较不同阈值算法
print("=" * 60)
print("mixed 模式下的阈值算法对比")
print("=" * 60)

for threshold_type in ["percentile", "standard_deviation", "interquartile", "gradient"]:
    chunks = run_semantic_chunker(
        test_doc,
        mode="mixed",
        regex=sentence_split_presets["mixed"],
        min_chunk_size=50,
        threshold_type=threshold_type,
    )
    print(f"\n[{threshold_type}]")
    print(f"块数量: {len(chunks)}")
    print(f"块长度列表: {[len(chunk.page_content) for chunk in chunks]}")


mixed 模式下的阈值算法对比

[percentile]
块数量: 3
块长度列表: [285, 87, 211]

[standard_deviation]
块数量: 1
块长度列表: [585]

[interquartile]
块数量: 1
块长度列表: [585]

[gradient]
块数量: 3
块长度列表: [284, 166, 133]


## 4. 检查内置分句规则与当前配置


In [10]:
# 检查项目当前支持的分句模式
from wxchatrag.chunking.semantic_splitter import SEMANTIC_SENTENCE_SPLIT_REGEX_PRESETS

print("项目内置的分句模式:")
for mode, regex in SEMANTIC_SENTENCE_SPLIT_REGEX_PRESETS.items():
    sentences = compact_sentences(re.split(regex, test_doc.page_content))
    print(f"\n[{mode}]")
    print(f"regex: {regex}")
    print(f"句子数: {len(sentences)}")
    print(f"前3句: {[s[:30] for s in sentences[:3]]}")

custom_regex = sentence_split_presets["custom"]
custom_sentences = compact_sentences(re.split(custom_regex, test_doc.page_content))
print("\n[custom]")
print(f"regex: {custom_regex}")
print(f"句子数: {len(custom_sentences)}")
print(f"前3句: {[s[:30] for s in custom_sentences[:3]]}")


项目内置的分句模式:

[english]
regex: (?<=[.?!])\s+
句子数: 1
前3句: ['人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常']

[chinese]
regex: (?<=[。！？；])|\n+
句子数: 16
前3句: ['人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常', '机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而', '深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作']

[mixed]
regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+
句子数: 16
前3句: ['人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常', '机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而', '深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作']

[custom]
regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+
句子数: 16
前3句: ['人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常', '机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而', '深度学习是机器学习的一个子集，它使用神经网络来模拟人脑的工作']


In [11]:
# 检查 SemanticChunker 与项目封装后的实际配置
from wxchatrag.chunking.semantic_splitter import SemanticSplitter

for mode in ["english", "chinese", "mixed"]:
    project_splitter = SemanticSplitter(
        embeddings=embeddings,
        chunk_size=50,
        chunk_overlap=0,
        similarity_threshold=0.5,
        breakpoint_threshold_type="percentile",
        sentence_split_mode=mode,
    )
    print(f"\n[{mode}]")
    print(f"项目封装 regex: {project_splitter.sentence_split_regex}")
    print(f"LangChain splitter regex: {project_splitter.splitter.sentence_split_regex}")

custom_project_splitter = SemanticSplitter(
    embeddings=embeddings,
    chunk_size=50,
    chunk_overlap=0,
    similarity_threshold=0.5,
    breakpoint_threshold_type="percentile",
    sentence_split_mode="custom",
    sentence_split_regex=sentence_split_presets["custom"],
)
print("\n[custom]")
print(f"项目封装 regex: {custom_project_splitter.sentence_split_regex}")
print(f"LangChain splitter regex: {custom_project_splitter.splitter.sentence_split_regex}")



[english]
项目封装 regex: (?<=[.?!])\s+
LangChain splitter regex: (?<=[.?!])\s+

[chinese]
项目封装 regex: (?<=[。！？；])|\n+
LangChain splitter regex: (?<=[。！？；])|\n+

[mixed]
项目封装 regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+
LangChain splitter regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+

[custom]
项目封装 regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+
LangChain splitter regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+


## 5. 测试当前实现的 SemanticSplitter


In [12]:
# 使用当前项目的 SemanticSplitter 封装进行模式对比
print("=" * 60)
print("测试当前实现的 SemanticSplitter")
print("=" * 60)

wrapped_results = {}
for mode in ["english", "chinese", "mixed", "custom"]:
    splitter_kwargs = {
        "embeddings": embeddings,
        "chunk_size": 50,
        "chunk_overlap": 0,
        "similarity_threshold": 0.5,
        "breakpoint_threshold_type": "percentile",
        "sentence_split_mode": mode,
    }
    if mode == "custom":
        splitter_kwargs["sentence_split_regex"] = sentence_split_presets["custom"]

    splitter = SemanticSplitter(**splitter_kwargs)
    chunks = splitter.split_documents([test_doc])
    wrapped_results[mode] = chunks

    print(f"\n[{mode}]")
    print(f"regex: {splitter.sentence_split_regex}")
    print(f"块数量: {len(chunks)}")
    print(f"块长度列表: {[len(chunk.page_content) for chunk in chunks]}")
    for i, chunk in enumerate(chunks[:3], 1):
        print(f"  块 {i}: {chunk.page_content[:80]!r}")


测试当前实现的 SemanticSplitter

[english]
regex: (?<=[.?!])\s+
块数量: 1
块长度列表: [576]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。\n\n机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'

[chinese]
regex: (?<=[。！？；])|\n+
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'
  块 2: '在人工智能的发展历程中，有几个重要的里程碑。 1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。 1956年，达特茅斯会议标志着人工智能学科'
  块 3: '随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。 深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。  然'

[mixed]
regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能够从数据中学习，而无需明确编'
  块 2: '在人工智能的发展历程中，有几个重要的里程碑。 1950年，图灵提出了著名的图灵测试，这是判断机器是否具有智能的标准。 1956年，达特茅斯会议标志着人工智能学科'
  块 3: '随着计算能力的提升和大数据的出现，人工智能在21世纪取得了突破性进展。 深度学习算法的成功应用，使得图像识别、语音识别和自然语言处理等领域取得了显著进步。  然'

[custom]
regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+
块数量: 3
块长度列表: [285, 87, 211]
  块 1: '人工智能（AI）是计算机科学的一个分支，旨在创建能够执行通常需要人类智能的任务的系统。  机器学习是人工智能的一个子集，它使计算机能

## 6. 检查 LangChain 版本和源代码


In [13]:
# 检查 LangChain 版本
import langchain_experimental
print(f"langchain_experimental 版本: {langchain_experimental.__version__}")

# 尝试查看 SemanticChunker 的源代码位置
print(f"\nSemanticChunker 源代码位置:")
print(inspect.getfile(SemanticChunker))

# 读取源代码的关键部分
try:
    source_file = inspect.getfile(SemanticChunker)
    print(f"\n读取源代码文件: {source_file}")
    
    with open(source_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    # 查找关键部分
    print("\n查找关键代码段 (包含 min_chunk_size, breakpoint, sentence):")
    found_lines = []
    for i, line in enumerate(lines, 1):
        if any(keyword in line.lower() for keyword in ['min_chunk_size', 'breakpoint', 'sentence_split']):
            found_lines.append((i, line.rstrip()))
    
    # 显示前20个匹配的行
    for i, line in found_lines[:20]:
        print(f"{i}: {line}")
except Exception as e:
    print(f"无法读取源代码: {e}")


langchain_experimental 版本: 0.4.1

SemanticChunker 源代码位置:
d:\worksoft\Anaconda3\envs\llmtorch\Lib\site-packages\langchain_experimental\text_splitter.py

读取源代码文件: d:\worksoft\Anaconda3\envs\llmtorch\Lib\site-packages\langchain_experimental\text_splitter.py

查找关键代码段 (包含 min_chunk_size, breakpoint, sentence):
88: BreakpointThresholdType = Literal[
91: BREAKPOINT_DEFAULTS: Dict[BreakpointThresholdType, float] = {
116:         breakpoint_threshold_type: BreakpointThresholdType = "percentile",
117:         breakpoint_threshold_amount: Optional[float] = None,
119:         sentence_split_regex: str = r"(?<=[.?!])\s+",
120:         min_chunk_size: Optional[int] = None,
125:         self.breakpoint_threshold_type = breakpoint_threshold_type
127:         self.sentence_split_regex = sentence_split_regex
128:         if breakpoint_threshold_amount is None:
129:             self.breakpoint_threshold_amount = BREAKPOINT_DEFAULTS[
130:                 breakpoint_threshold_type
133:             self.bre

## 7. 使用真实 PDF 文档测试不同分句模式


In [14]:
# 加载真实PDF文档
from wxchatrag.wxhub_loader import iter_pdf_paths, load_pdf_documents, build_channel_indexes

settings = get_settings()
wxhub_root = settings.wxhub_root
glob_pattern = settings.pdf_glob_pattern or f"*/{settings.pdf_subdir_name}/*.pdf"

pdf_paths = iter_pdf_paths(wxhub_root, glob_pattern)

if pdf_paths:
    first_pdf_path = pdf_paths[0]
    print(f"使用PDF: {first_pdf_path.name}")

    channel_indexes = build_channel_indexes(wxhub_root)
    pdf_docs = load_pdf_documents(
        pdf_paths=[first_pdf_path],
        channel_indexes=channel_indexes
    )

    if pdf_docs:
        all_content = "\n\n".join([doc.page_content for doc in pdf_docs])
        real_doc = Document(
            page_content=all_content,
            metadata=pdf_docs[0].metadata
        )

        print(f"文档长度: {len(real_doc.page_content)} 字符")
        print("\n测试语义切分:")

        for mode in ["english", "chinese", "mixed", "custom"]:
            splitter_kwargs = {
                "embeddings": embeddings,
                "chunk_size": 200,
                "chunk_overlap": 0,
                "similarity_threshold": 0.5,
                "breakpoint_threshold_type": "percentile",
                "sentence_split_mode": mode,
            }
            if mode == "custom":
                splitter_kwargs["sentence_split_regex"] = sentence_split_presets["custom"]

            splitter = SemanticSplitter(**splitter_kwargs)
            chunks = splitter.split_documents([real_doc])
            print(f"\n[{mode}]")
            print(f"regex: {splitter.sentence_split_regex}")
            print(f"块数量: {len(chunks)}")
            print(f"前5块长度: {[len(chunk.page_content) for chunk in chunks[:5]]}")
            if chunks:
                print(f"首块预览: {chunks[0].page_content[:120]!r}")
else:
    print("未找到PDF文件，跳过此测试")


使用PDF: 2026-02-19-人工智能时代真正重要的技能：你的品味.pdf
文档长度: 9469 字符

测试语义切分:

[english]
regex: (?<=[.?!])\s+
块数量: 1
前5块长度: [9469]
首块预览: '人工智能时代真正重要的技能：你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标 \x00️ ， \x0e ，每天才能第一时间接收到更新 在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的'

[chinese]
regex: (?<=[。！？；])|\n+
块数量: 13
前5块长度: [3185, 358, 1023, 505, 317]
首块预览: '人工智能时代真正重要的技能：你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标 \x00️ ， \x0e ，每天才能第一时间接收到更新 在人工智能时代，品味将变得更加重要。 当任何人都能制作任何东西时，真正'

[mixed]
regex: (?<=[。！？；])|(?<=[.?!])\s+|\n+
块数量: 13
前5块长度: [535, 2649, 358, 1023, 823]
首块预览: '人工智能时代真正重要的技能：你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标 \x00️ ， \x0e ，每天才能第一时间接收到更新 在人工智能时代，品味将变得更加重要。 当任何人都能制作任何东西时，真正'

[custom]
regex: (?<=[。！？；：])|(?<=[.?!:;])\s+|\n+
块数量: 13
前5块长度: [2883, 311, 358, 1026, 506]
首块预览: '人工智能时代真正重要的技能： 你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标 \x00️ ， \x0e ，每天才能第一时间接收到更新 在人工智能时代，品味将变得更加重要。 当任何人都能制作任何东西时，真'


## 8. 结果解读

运行本 notebook 时，重点关注以下现象：

1. `english` 模式通常只会得到 1 句或极少句，因为默认英文规则无法正确拆分中文段落。
2. `chinese` 与 `mixed` 模式应该显著增加句子数，并让语义切分有机会产生多个块。
3. `mixed` 更适合公众号/PDF 场景，因为正文里常常混有英文标点、换行和中英文混排。
4. `custom` 用于进一步兼容特殊符号，例如冒号、分号、OCR 产生的断行等。

### 建议观察指标

- 分句数是否明显增加
- `SemanticChunker` 的块数量是否从 1 变为多块
- 不同模式下块长度是否更均衡
- 真实 PDF 中 `mixed` 是否明显优于 `english`


## 9. 建议测试顺序

1. 先运行第 3 节，确认 `english/chinese/mixed/custom` 的原始分句数差异。
2. 再运行第 5 节，确认项目封装后的 `SemanticSplitter` 是否正确把 `sentence_split_regex` 传给 LangChain。
3. 最后运行第 7 节，用真实 PDF 验证 `mixed` 是否能把原本的 1 块拆成多块。

### 推荐配置

如果你的数据以中文公众号 PDF 为主，建议优先测试：

```python
sentence_split_mode="mixed"
```

如果文档 OCR 噪声比较多，建议继续试：

```python
sentence_split_mode="custom"
sentence_split_regex=r"(?<=[。！？；：])|(?<=[.?!:;])\s+|\n+"
```
